# Truth-matching analysis -- ICARUSNuGraphAnalyzer

## Run command

```
lar -c nugraph_analyser.fcl \
    -s /exp/icarus/app/users/sdey2/icaruscode-v10_06_00_04p04_production/nugraphProduction/localRunTest/MPVMPRTrainingDataset_MiniProductionTest/stage1-2dmodel-fixednorm-trainsample.root \
    -n 5
```

## Output

`hist_nugraph_analyser_truth_trainingsample.root`, tree
`NuGraphAnaCryoE/NuGraphHitOutput` (+ `NuGraphEventOutput`, `NuGraphEdepOutput`,
`NuGraphSliceOutput`, `NuGraphPFPOutput`). 5 events, 53,292 hits, same
underlying stage1 sample used for the RecoFix comparison run.

## What had to change to get truth info working

Starting point was `ICARUSNuGraphAnalyzer_module.cc` with the three RecoFix
bugfixes already applied (uninitialized `icluster`/`ipfp`, dead
`NuGraphEventOutput` tree, stale NG scores on invalid-output hits -- see
the separate RecoFix notebook for those). On top of that, to get truth
info flowing end to end:

**Code (`CMakeLists.txt`):** the module needed three more libraries linked
that the original build never required, since it now touches
`BackTrackerService`/`ParticleInventoryService`/`MCTruth`:
```cmake
cet_build_plugin(ICARUSNuGraphAnalyzer art::EDAnalyzer
    LIBRARIES PRIVATE
    art_root_io::TFileService_service
    ROOT::Tree
    lardata::RecoBaseProxy
    lardataobj::RecoBase
    nusimdata::SimulationBase
    larsim::MCCheater_BackTrackerService_service
    larsim::MCCheater_ParticleInventoryService_service
    lardata::DetectorClocksService
)
```
(`ROOT::Physics`, needed for an earlier `TDatabasePDG`-based charge lookup,
was tried and dropped -- it wouldn't link cleanly for this plugin type for
reasons not worth chasing further, so that lookup was reimplemented as a
small hardcoded PDG list instead of a general ROOT database call.)

**Code (`#include`s):** `lar::providerFrom<>()` needed
`larcore/CoreUtils/ServiceUtil.h`, which the file didn't originally include.

**fcl:** the plain original fcl has no truth-related services configured at
all. Needed:
- `#include "services_icarus_simulation.fcl"` (defines
  `icarus_backtracking_services` -- `services_common_icarus.fcl` alone
  doesn't have it)
- `@table::icarus_common_services` + `@table::icarus_backtracking_services`
  in the `services` block
- An explicit `ParticleInventoryService` block (`G4ModuleLabel: "largeant"`, etc.)
- `TruthLabel: "generator"` on the analyzer itself
- `services.BackTrackerService.BackTracker.G4ModuleLabel`/`SimChannelModuleLabel`
  overrides at the bottom of the file
- A late `services.TFileService.fileName` override, since
  `icarus_common_services` turned out to define its own `TFileService` with
  a SAM-style auto-generated name that silently clobbered the literal
  filename set earlier in the same block (FHiCL last-assignment-wins).

In [ ]:
#!pip install scikit-learn

In [ ]:
import uproot
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import cmcrameri.cm as cmc
from sklearn.metrics import confusion_matrix

%matplotlib inline

# --- style: serif font, no gridlines, Managua colourscheme ---
mpl.rcParams["font.family"] = "serif"
mpl.rcParams["axes.grid"] = False
mpl.rcParams["axes.spines.top"] = False
mpl.rcParams["axes.spines.right"] = False

MANAGUA = cmc.managua
MANAGUA_R = cmc.managua_r
def managua_discrete(n, reverse=False):
    cmap = MANAGUA_R if reverse else MANAGUA
    return [cmap(x) for x in np.linspace(0.08, 0.92, n)]

score_cols = ["MIP", "HIP", "shower", "michel", "diffuse"]
class_labels = ["MIP", "HIP", "Shower", "Michel", "Diffuse"]

## Load all three files

- **truth**: full truth-matching + RecoFix (`NuGraphAnaCryoE`)
- **recofix**: RecoFix bugfixes only, no truth (`NuGraphAnaCryoE` = original,
  `NuGraphAnaCryoERecoFix` = fixed)
- **plain**: original module, unmodified, no truth, no fixes

All three point at the *same* stage1 input and the *same* 5 events -- this
is what makes the cross-file comparison below meaningful rather than
comparing apples to oranges.

In [ ]:
paths = {
    "truth":   "hist_nugraph_analyser_truth_trainingsample.root",
    "recofix": "hist_nugraph_analyser_recofix_trainingsample.root",
    "plain":   "hist_nugraph_analyser_trainingsample.root",
}

truth_cols = ["event", "truth_semantic", "truth_isnu", "truth_g4id", "has_ng_output", *score_cols]
basic_cols = [c for c in truth_cols if c not in ("truth_semantic", "truth_isnu", "truth_g4id")]

data = {}
f = uproot.open(paths["truth"])
data["truth"] = f["NuGraphAnaCryoE/NuGraphHitOutput"].arrays(truth_cols, library="pd")

f = uproot.open(paths["recofix"])
avail = f["NuGraphAnaCryoERecoFix/NuGraphHitOutput"].keys()
cols = [c for c in basic_cols if c in avail]
data["recofix"] = f["NuGraphAnaCryoERecoFix/NuGraphHitOutput"].arrays(cols, library="pd")
data["recofix_orig"] = f["NuGraphAnaCryoE/NuGraphHitOutput"].arrays(score_cols, library="pd")

f = uproot.open(paths["plain"])
data["plain"] = f["NuGraphAnaCryoE/NuGraphHitOutput"].arrays(score_cols, library="pd")

for k, v in data.items():
    print(f"{k:14s} {len(v)} hits")

## Check 1 -- truth-match stats

How much of the truth-matching machinery actually worked, on the one file
that has it.

In [ ]:
hits = data["truth"]
n = len(hits)

summary = pd.DataFrame({
    "check": [
        "hits with valid NG output",
        "hits with a truth match (truth_g4id >= 0)",
        "hits from the neutrino interaction (dominant contributor)",
        "broken particle-chain hits (truth_semantic == -2)",
        "invisible-particle hits (truth_semantic == 5)",
    ],
    "fraction": [
        (hits["has_ng_output"]==1).mean(),
        (hits["truth_g4id"]>=0).mean(),
        (hits["truth_isnu"]==1).mean(),
        (hits["truth_semantic"]==-2).mean(),
        (hits["truth_semantic"]==5).mean(),
    ],
    "n_hits": [
        int((hits["has_ng_output"]==1).sum()),
        int((hits["truth_g4id"]>=0).sum()),
        int((hits["truth_isnu"]==1).sum()),
        int((hits["truth_semantic"]==-2).sum()),
        int((hits["truth_semantic"]==5).sum()),
    ],
})
summary["fraction"] = summary["fraction"].map(lambda x: f"{x:.2%}")
summary

## Check 2 -- semantic label distributions

Three things bundled together here:

1. **True label distribution** (this file only -- needs truth)
2. **Predicted label distribution, all three files** -- expectation stated
   up front: this should be **identical** across `truth`/`recofix`/`plain`,
   since none of the RecoFix or truth-matching additions touch the
   `semantic->at(i)` readout that produces MIP/HIP/shower/michel/diffuse --
   they only add bookkeeping and truth columns alongside it. If these
   *don't* match, that's a real problem worth chasing, not an expected
   side-effect of any of this work.
3. **Overlay against Giulia's training-truth reference histogram**
   (thesis Fig. 4.10, `final_sample.gnn.h5`). Those numbers are read off
   the published plot by eye, not the underlying data file -- treat as
   approximate until replaced with exact values if you have access to the
   original histogram.

In [ ]:
def predicted_dist(df):
    c = np.bincount(df[score_cols].values.argmax(axis=1), minlength=5)
    return c / c.sum()

pred = {
    "truth (this run)": predicted_dist(data["truth"]),
    "recofix": predicted_dist(data["recofix"]),
    "recofix (orig tree)": predicted_dist(data["recofix_orig"]),
    "plain": predicted_dist(data["plain"]),
}

pred_df = pd.DataFrame(pred, index=class_labels).T
pred_df["max abs diff vs plain"] = (pred_df[class_labels].sub(pred_df.loc["plain", class_labels], axis=1)
                                     .abs().max(axis=1))
pred_df.round(4)

In [ ]:
# --- true label distribution (this run) ---
plot_labels = {-1: "Noise", 0: "MIP", 1: "HIP", 2: "Shower", 3: "Michel", 4: "Diffuse"}
values = sorted(plot_labels.keys())
true_freq = np.array([(hits["truth_semantic"] == v).sum() for v in values]) / len(hits)

# --- predicted label distribution (this run, argmax) ---
pred_freq = predicted_dist(data["truth"])
pred_freq_full = np.concatenate([[np.nan], pred_freq])  # align to the -1..4 axis (no "noise" prediction)

# --- Giulia's reference (Fig. 4.10, read off by eye -- REPLACE with exact values if available) ---
giulia_freq = np.array([0.045, 0.297, 0.190, 0.430, 0.013, 0.035])

x = np.arange(len(values))
width = 0.27
colors = managua_discrete(3)

fig, ax = plt.subplots(figsize=(8, 6))
ax.bar(x - width, true_freq, width, label="True label (this run)", color=colors[0])
ax.bar(x, pred_freq_full, width, label="Predicted label (argmax, this run)", color=colors[1])
ax.bar(x + width, giulia_freq, width, label="Giulia's training truth (Fig. 4.10, approx.)", color=colors[2])

ax.set_xticks(x)
ax.set_xticklabels([plot_labels[v] for v in values])
ax.set_xlabel("semantic label")
ax.set_ylabel("frequency")
ax.set_title("Semantic label distribution: true vs. predicted vs. reference", fontsize=12)
ax.legend(frameon=False)
plt.tight_layout()
plt.savefig("semantic_distribution_comparison.png", dpi=350)
plt.show()

## Check 3 -- confusion matrix

Filters to hits that are (a) valid NG predictions and (b) one of the 5
trained classes -- excludes `-1`/`-2` (no usable truth) and `5` (invisible,
has truth but isn't a class NuGraph was ever trained to predict).

In [ ]:
valid = (hits["has_ng_output"] == 1) & (hits["truth_semantic"].between(0, 4))
cm_hits = hits[valid].copy()
cm_hits["assigned"] = cm_hits[score_cols].values.argmax(axis=1)

print(f"hits used for confusion matrix: {len(cm_hits)} / {len(hits)} ({len(cm_hits)/len(hits):.1%})")

cm = confusion_matrix(cm_hits["truth_semantic"], cm_hits["assigned"], labels=range(5))
recall = cm / cm.sum(axis=1, keepdims=True)
precision = cm / cm.sum(axis=0, keepdims=True)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

for ax, mat, title in zip(axes, [recall, precision], ["Semantic Recall", "Semantic Precision"]):
    im = ax.imshow(mat, cmap=MANAGUA, vmin=0, vmax=1)
    for i in range(5):
        for j in range(5):
            val = mat[i, j]
            txt_color = "white" if (val > 0.6 or val < 0.15) else "black"
            ax.text(j, i, f"{val:.2f}", ha="center", va="center", color=txt_color, fontsize=11)
    ax.set_xticks(range(5)); ax.set_xticklabels(class_labels)
    ax.set_yticks(range(5)); ax.set_yticklabels(class_labels)
    ax.set_xlabel("Assigned label"); ax.set_ylabel("True label")
    ax.set_title(title, fontsize=13)
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="fraction")

plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=350)
plt.show()

In [ ]:
per_class = pd.DataFrame({
    "class": class_labels,
    "n_true": cm.sum(axis=1),
    "n_predicted": cm.sum(axis=0),
    "recall (diag)": np.diag(recall).round(3),
    "precision (diag)": np.diag(precision).round(3),
})
per_class

## A note on statistical rigor with 5 events

Two honest caveats worth sitting with before trusting the numbers above as
precise:

**1. Hits are not independent samples.** 53,292 hits sounds like a lot, but
they come from only **5 events** -- hits within one track or shower are
massively correlated (they're literally the same particle's energy
deposits). Treating each hit as an independent Bernoulli trial and slapping
a binomial confidence interval on each confusion-matrix cell would
understate the true uncertainty, sometimes drastically. The right unit of
"how much data do we have" here is closer to 5, not 53,292.

**2. What a rigorous estimate looks like given that.** An **event-level
bootstrap** -- resample the 5 events with replacement, recompute the full
confusion matrix on each resample, repeat many times, look at the spread --
respects the actual clustering structure instead of pretending hits are
independent. It's still fundamentally limited by only having 5 events to
draw from (it can't manufacture information that isn't there), but the
width of the resulting interval is a much more honest statement of
uncertainty than a per-hit binomial CI would give you.

Below: per-event recall to see whether the headline numbers are being
driven by one or two events, then the event-level bootstrap itself.

In [ ]:
# --- per-event breakdown ---
events = hits  # already has an "event" column loaded above

per_event_recall = {}
for ev in sorted(events["event"].unique()):
    sub = events[events["event"] == ev]
    v = (sub["has_ng_output"] == 1) & (sub["truth_semantic"].between(0, 4))
    sub = sub[v]
    if len(sub) == 0:
        continue
    assigned = sub[score_cols].values.argmax(axis=1)
    cm_ev = confusion_matrix(sub["truth_semantic"], assigned, labels=range(5))
    with np.errstate(invalid="ignore", divide="ignore"):
        recall_ev = np.diag(cm_ev) / cm_ev.sum(axis=1)
    per_event_recall[f"evt {ev}"] = recall_ev

per_event_df = pd.DataFrame(per_event_recall, index=class_labels).T
per_event_df.round(3)

In [ ]:
# --- event-level (cluster) bootstrap --- [fine for the 5 ev test]
rng = np.random.default_rng(0)
event_ids = sorted(events["event"].unique())
n_boot = 1000

boot_recall = np.full((n_boot, 5), np.nan)
boot_precision = np.full((n_boot, 5), np.nan)

for b in range(n_boot):
    sample_events = rng.choice(event_ids, size=len(event_ids), replace=True)
    parts = [events[events["event"] == ev] for ev in sample_events]
    boot_df = pd.concat(parts, ignore_index=True)

    v = (boot_df["has_ng_output"] == 1) & (boot_df["truth_semantic"].between(0, 4))
    boot_df = boot_df[v]
    assigned = boot_df[score_cols].values.argmax(axis=1)
    cm_b = confusion_matrix(boot_df["truth_semantic"], assigned, labels=range(5))

    with np.errstate(invalid="ignore", divide="ignore"):
        boot_recall[b] = np.diag(cm_b) / cm_b.sum(axis=1)
        boot_precision[b] = np.diag(cm_b) / cm_b.sum(axis=0)

recall_lo, recall_hi = np.nanpercentile(boot_recall, [16, 84], axis=0)
prec_lo, prec_hi = np.nanpercentile(boot_precision, [16, 84], axis=0)
recall_point = np.diag(recall)
prec_point = np.diag(precision)

boot_summary = pd.DataFrame({
    "class": class_labels,
    "recall": recall_point.round(3),
    "recall 68% CI (event bootstrap)": [f"[{lo:.2f}, {hi:.2f}]" for lo, hi in zip(recall_lo, recall_hi)],
    "precision": prec_point.round(3),
    "precision 68% CI (event bootstrap)": [f"[{lo:.2f}, {hi:.2f}]" for lo, hi in zip(prec_lo, prec_hi)],
})
boot_summary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5.5))
colors = managua_discrete(5)

for ax, point, lo, hi, title in zip(
    axes, [recall_point, prec_point], [recall_lo, prec_lo], [recall_hi, prec_hi],
    ["Recall", "Precision"],
):
    yerr = np.vstack([point - lo, hi - point])
    yerr = np.clip(yerr, 0, None)
    ax.bar(class_labels, point, color=colors, yerr=yerr, capsize=4,
           error_kw={"elinewidth": 1.3, "ecolor": "black"})
    ax.set_ylim(0, 1)
    ax.set_ylabel(title.lower())
    ax.set_title(f"{title} with 68% event-level bootstrap CI", fontsize=12)

plt.tight_layout()
plt.savefig("bootstrap_uncertainty.png", dpi=350)
plt.show()

**Reading this plot**: classes with a wide bootstrap band (likely Michel
and Diffuse, given they're your rarest true classes) mean the point
estimate is genuinely unstable given only 5 events -- a different 5-event
sample could plausibly give a very different number. That's not a flaw in
the analyzer, it's just what "5 events" means statistically, and it's the
honest caveat to put next to any Michel/Diffuse precision number before
presenting it as a stable result.

### Exact version -- all 126 possible resamples, not a random 1000-draw approximation

With exactly 5 events, there are only `C(9,5) = 126` distinct possible
resamples-with-replacement of size 5. 1000 random draws necessarily hits
many of those 126 outcomes more than once -- the cell above is a Monte
Carlo *approximation* of something small enough to just compute exactly.

Doing it exactly means enumerating all 126 multisets and weighting each by
its true probability under sampling-with-replacement (a multiset with all
5 events distinct, e.g. `{1,2,3,4,5}`, is far more likely to be drawn than
one with all 5 the same, e.g. `{1,1,1,1,1}` -- naive equal-weighting of the
126 outcomes would get this wrong). The two versions should agree closely
if 1000 draws was already enough; if they don't, that's a sign the random
version hadn't converged.

In [ ]:
import itertools
from math import factorial

def multiset_prob(combo, n_events):
    """Exact probability of drawing this specific multiset (as a sorted
    tuple) when sampling len(combo) events with replacement from n_events
    equally-likely events."""
    k = len(combo)
    counts = pd.Series(combo).value_counts()
    multinom_coeff = factorial(k)
    for c in counts:
        multinom_coeff //= factorial(c)
    return multinom_coeff / (n_events ** k)

def weighted_percentile(values, weights, percentiles):
    order = np.argsort(values)
    values, weights = np.asarray(values)[order], np.asarray(weights)[order]
    cum_weights = np.cumsum(weights) - 0.5 * weights
    cum_weights /= weights.sum()
    return np.interp(np.array(percentiles) / 100.0, cum_weights, values)

all_combos = list(itertools.combinations_with_replacement(event_ids, len(event_ids)))
print(f"n distinct resamples: {len(all_combos)} (vs {n_boot} random draws used above)")

exact_recall = np.full((len(all_combos), 5), np.nan)
exact_precision = np.full((len(all_combos), 5), np.nan)
exact_weights = np.zeros(len(all_combos))

for i, combo in enumerate(all_combos):
    exact_weights[i] = multiset_prob(combo, len(event_ids))
    parts = [events[events["event"] == ev] for ev in combo]
    combo_df = pd.concat(parts, ignore_index=True)

    v = (combo_df["has_ng_output"] == 1) & (combo_df["truth_semantic"].between(0, 4))
    combo_df = combo_df[v]
    assigned = combo_df[score_cols].values.argmax(axis=1)
    cm_c = confusion_matrix(combo_df["truth_semantic"], assigned, labels=range(5))

    with np.errstate(invalid="ignore", divide="ignore"):
        exact_recall[i] = np.diag(cm_c) / cm_c.sum(axis=1)
        exact_precision[i] = np.diag(cm_c) / cm_c.sum(axis=0)

print(f"weights sum to {exact_weights.sum():.6f} (should be 1.0)")

In [ ]:
exact_recall_lo, exact_recall_hi = np.full(5, np.nan), np.full(5, np.nan)
exact_prec_lo, exact_prec_hi = np.full(5, np.nan), np.full(5, np.nan)

for c in range(5):
    mask = ~np.isnan(exact_recall[:, c])
    if mask.sum() > 0:
        w = exact_weights[mask]
        exact_recall_lo[c], exact_recall_hi[c] = weighted_percentile(exact_recall[mask, c], w, [16, 84])
    mask = ~np.isnan(exact_precision[:, c])
    if mask.sum() > 0:
        w = exact_weights[mask]
        exact_prec_lo[c], exact_prec_hi[c] = weighted_percentile(exact_precision[mask, c], w, [16, 84])

compare = pd.DataFrame({
    "class": class_labels,
    "recall 68% CI (n=1000 MC)":    [f"[{lo:.2f}, {hi:.2f}]" for lo, hi in zip(recall_lo, recall_hi)],
    "recall 68% CI (exact, n=126)": [f"[{lo:.2f}, {hi:.2f}]" for lo, hi in zip(exact_recall_lo, exact_recall_hi)],
    "precision 68% CI (n=1000 MC)":    [f"[{lo:.2f}, {hi:.2f}]" for lo, hi in zip(prec_lo, prec_hi)],
    "precision 68% CI (exact, n=126)": [f"[{lo:.2f}, {hi:.2f}]" for lo, hi in zip(exact_prec_lo, exact_prec_hi)],
})
compare